# VMTF-Net - Kaggle / Colab quickstart

1. Settings → Accelerator: **GPU T4 x2** (or P100). Internet **On** (or attach PhoBERT/ViT as Kaggle Models, see step 4).
2. Add your dataset (CSV splits + images) with *Add Input*. Edit the paths in the next cell.
3. Long runs: if the session stops, re-run the training cell with `RESUME = True`.

In [ ]:
import os

IS_KAGGLE = os.path.exists('/kaggle')
WORK = '/kaggle/working' if IS_KAGGLE else '/content'

# ---- EDIT THESE PATHS ----
DATA_DIR  = '/kaggle/input/YOUR-DATASET'      # contains train.csv / val.csv / test.csv
IMAGE_DIR = f'{DATA_DIR}/images'
TEXT_MODEL   = 'vinai/phobert-base'           # or a local path, e.g. /kaggle/input/phobert-base/...
VISUAL_MODEL = 'google/vit-base-patch16-224'
EXP_DIR = f'{WORK}/checkpoints/proposed_seed42'
RESUME = False

!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd {WORK}
!git clone -q https://github.com/kascik04/vietnamese-multimodal-fake-news-detection.git || (cd vietnamese-multimodal-fake-news-detection && git pull -q)
%cd {WORK}/vietnamese-multimodal-fake-news-detection
!pip install -q pyvi imagehash

In [ ]:
# Offline unit + end-to-end tests (tiny random backbones, ~1 minute)
!python test_step1.py && python test_step2.py && python test_step3.py && python test_step4.py && python test_step5.py

In [ ]:
# ReINTEL: convert to the project schema, then de-duplicate + stratified split into /kaggle/working/data
RAW_DIR = '/kaggle/input/YOUR-REINTEL-DATASET'   # contains public_train.csv (+ images/)
!python scripts/convert_reintel.py --input {RAW_DIR}/public_train.csv --image_dir {RAW_DIR}/images --output {WORK}/data/reintel_vmtf.csv
!python scripts/validate_and_preprocess_dataset.py --input_csv {WORK}/data/reintel_vmtf.csv --image_dir {RAW_DIR}/images \n    --output_clean_csv {WORK}/data/clean.csv --output_dir {WORK}/data --split_by group
DATA_DIR, IMAGE_DIR = f'{WORK}/data', f'{RAW_DIR}/images'   # train/val/test.csv are now here

In [ ]:
# Optional, once: downscale images to speed up data loading (Kaggle inputs are read-only)
!python scripts/resize_images.py --input_dir {IMAGE_DIR} --output_dir {WORK}/images --max_side 320
IMAGE_DIR = f'{WORK}/images'

In [ ]:
# RESUME=True continues an interrupted run. A finished run refuses to be overwritten:
# change EXP_DIR or add --overwrite to the command.
resume_flag = f'--resume {EXP_DIR}/last_checkpoint.pt' if RESUME and os.path.exists(f'{EXP_DIR}/last_checkpoint.pt') else ''

!python -m vmtf_net.train \
    --train_csv {DATA_DIR}/train.csv --val_csv {DATA_DIR}/val.csv --test_csv {DATA_DIR}/test.csv \
    --image_dir {IMAGE_DIR} --cache_dir {WORK}/cache \
    --text_model_name {TEXT_MODEL} --visual_model_name {VISUAL_MODEL} \
    --fusion_type hybrid_proposed --consistency_lambda 0.1 --social_preset basic \
    --batch_size 16 --epochs 10 --lr 2e-5 --head_lr 1e-4 --seed 42 \
    --checkpoint_dir {EXP_DIR} --no_tqdm {resume_flag}

In [ ]:
import json
import pandas as pd

history = pd.read_csv(f'{EXP_DIR}/history.csv')
display(history)
ax = history.plot(x='epoch', y=['train_f1_macro', 'val_f1_macro'], marker='o', title='Macro-F1')

results = json.load(open(f'{EXP_DIR}/results.json'))
print(json.dumps(results['test']['metrics'], indent=2))
print(json.dumps(results['test'].get('confidence_intervals_95', {}), indent=2))

## Ablations / baselines
Change the flags of the training cell (and `EXP_DIR`), e.g. `--fusion_type late_concat --consistency_lambda 0`, `--no_gate`, `--no_cross_attention`, `--no_social`, `--fusion_type text_only`.
Multi-seed: `python scripts/run_seeds.py --seeds 13 42 2024 --out_dir {WORK}/checkpoints/proposed -- <train flags>`.
Significance: `python scripts/significance_test.py --a <baseline>/test_predictions.csv --b <proposed>/test_predictions.csv`.

`last_checkpoint.pt` (~3GB) is deleted automatically when a run finishes, so the output only keeps `best_vmtf_model.pt` + JSON/CSV files. With many experiments, `best_vmtf_model.pt` (~0.9GB each) can still fill the 20GB `/kaggle/working` limit: download or delete old ones.